# LLM Summarizer — Ollama + llama3.2

**Objetivo:** Demostrar el pipeline completo: anomalía detectada → resumen accionable en lenguaje natural.

**Prerrequisito:** `ollama serve` corriendo con `llama3.2` instalado.

```bash
ollama pull llama3.2
ollama serve
```

Si Ollama no está disponible, el notebook muestra el fallback graceful.

In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import json
from datetime import timedelta
from pathlib import Path

from src.data.loader import load_bgl_logs
from src.data.preprocessor import add_severity_score, train_test_split_temporal
from src.features.engineering import load_features
from src.models.detector import AnomalyDetector
from src.llm.summarizer import LogSummarizer, build_anomaly_context

MODEL_PATH = Path('../models/saved/lof_v1.joblib')
FEATURES_PATH = Path('../data/processed/features_train.parquet')

In [3]:
summarizer = LogSummarizer(model='llama3.2')
print(f'Ollama disponible: {summarizer.is_available}')

2026-06-09 09:40:01 [info     ] ollama_connected               model=llama3.2
Ollama disponible: True


## 1. Cargar anomalías detectadas

In [4]:
detector = AnomalyDetector.load(MODEL_PATH)
df_feat = load_features(FEATURES_PATH)

feature_cols = [c for c in df_feat.columns if c not in {'timestamp', 'node', 'is_anomaly'}]
_, test_df = train_test_split_temporal(df_feat, test_fraction=0.2)

X_test = test_df[feature_cols].fillna(0).values
y_pred = detector.predict(X_test)
scores = detector.score_samples(X_test)

test_df = test_df.copy()
test_df['y_pred'] = y_pred
test_df['anomaly_score'] = scores

detected = test_df[test_df['y_pred'] == 1].nlargest(20, 'anomaly_score')
print(f'Anomalías detectadas (top 20 por score): {len(detected)}')

Anomalías detectadas (top 20 por score): 20


In [5]:
# Cargar raw con mismo nrows que features_train (400K) para alinear timestamps
df_raw = load_bgl_logs(Path('../data/raw/BGL.log'), nrows=400_000)
df_raw = add_severity_score(df_raw)
_, df_raw_test = train_test_split_temporal(df_raw)
print(f'Logs raw en test: {len(df_raw_test):,}')
print(f'Rango: {df_raw_test["timestamp"].min()} → {df_raw_test["timestamp"].max()}')

Logs raw en test: 80,000
Rango: 2005-06-11 21:20:08.343252 → 2005-06-13 17:45:20.517798


## 2. Generar resúmenes LLM para las top anomalías

In [9]:
summaries = []

for _, row in detected.head(5).iterrows():
    ts = row['timestamp']
    node = str(row['node'])

    window = df_raw_test[
        (df_raw_test['timestamp'] >= ts - timedelta(minutes=5)) &
        (df_raw_test['timestamp'] <= ts + timedelta(minutes=5))
    ]

    if len(window) == 0:
        continue

    context = build_anomaly_context(
        df_window=window,
        anomaly_score=float(row['anomaly_score']),
        node=node,
        timestamp=str(ts),
    )

    result = summarizer.summarize(context)
    result['timestamp'] = str(ts)
    result['node'] = node
    result['anomaly_score'] = float(row['anomaly_score'])
    summaries.append(result)

print(f'Resúmenes generados: {len(summaries)}')

2026-06-09 09:44:21 [info     ] anomaly_summarized             node=R30-M0-N9-C:J16-U01 response_time_ms=27038.3 severity=UNKNOWN
2026-06-09 09:44:28 [info     ] anomaly_summarized             node=R30-M0-N9-C:J16-U01 response_time_ms=5986.3 severity=UNKNOWN
2026-06-09 09:44:34 [info     ] anomaly_summarized             node=R30-M0-N9-C:J16-U01 response_time_ms=6619.8 severity=UNKNOWN
2026-06-09 09:44:41 [info     ] anomaly_summarized             node=R30-M0-N9-C:J16-U01 response_time_ms=6835.3 severity=UNKNOWN
2026-06-09 09:44:48 [info     ] anomaly_summarized             node=R30-M0-N9-C:J16-U01 response_time_ms=6488.1 severity=UNKNOWN
Resúmenes generados: 5


## 3. Ejemplos de output del LLM

In [7]:
for i, s in enumerate(summaries, 1):
    print('=' * 60)
    print(f'ANOMALÍA #{i}')
    print(f'Nodo: {s["node"]} | Timestamp: {s["timestamp"]}')
    print(f'Score: {s["anomaly_score"]:.4f} | Tiempo LLM: {s.get("response_time_ms", 0):.0f}ms')
    print()
    print(f'Resumen:            {s.get("resumen", "N/A")}')
    print(f'Severidad:          {s.get("severidad", "N/A")}')
    print(f'Causa probable:     {s.get("causa_probable", "N/A")}')
    print(f'Acción recomendada: {s.get("accion_recomendada", "N/A")}')
    print()

ANOMALÍA #1
Nodo: R30-M0-N9-C:J16-U01 | Timestamp: 2005-06-11 23:32:33.186731
Score: 69004232.0000 | Tiempo LLM: 87ms

Resumen:            LLM not available — summary skipped.
Severidad:          UNKNOWN
Causa probable:     LLM not available
Acción recomendada: Review logs manually

ANOMALÍA #2
Nodo: R30-M0-N9-C:J16-U01 | Timestamp: 2005-06-11 23:32:33.286508
Score: 69004232.0000 | Tiempo LLM: 2ms

Resumen:            LLM not available — summary skipped.
Severidad:          UNKNOWN
Causa probable:     LLM not available
Acción recomendada: Review logs manually

ANOMALÍA #3
Nodo: R30-M0-N9-C:J16-U01 | Timestamp: 2005-06-11 23:32:32.751022
Score: 46454992.0000 | Tiempo LLM: 2ms

Resumen:            LLM not available — summary skipped.
Severidad:          UNKNOWN
Causa probable:     LLM not available
Acción recomendada: Review logs manually

ANOMALÍA #4
Nodo: R30-M0-N9-C:J16-U01 | Timestamp: 2005-06-11 23:04:00.058904
Score: 45680456.0000 | Tiempo LLM: 3ms

Resumen:            LLM not avai

## 4. Métricas del summarizer

In [8]:
if summaries:
    times = [s.get('response_time_ms', 0) for s in summaries]
    severities = [s.get('severidad', 'UNKNOWN') for s in summaries]

    print('Tiempo de respuesta LLM:')
    print(f'  Media:   {np.mean(times):.0f}ms')
    print(f'  Mediana: {np.median(times):.0f}ms')
    print(f'  p95:     {np.percentile(times, 95):.0f}ms')
    print()
    print('Distribución de severidades:')
    for sev, count in pd.Series(severities).value_counts().items():
        print(f'  {sev}: {count}')

Tiempo de respuesta LLM:
  Media:   19ms
  Mediana: 2ms
  p95:     70ms

Distribución de severidades:
  UNKNOWN: 5
